<a href="https://colab.research.google.com/github/BariscanTosyali/CryptoVisionAI/blob/main/work/notebooks/w08_capstone_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Capstone — Content Decay Ranking & Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BariscanTosyali/flyrank-ml-internship/blob/main/work/notebooks/w08_capstone_model.ipynb?flush_cache=true)

This capstone builds a machine learning ranking model to identify organic search content decay without target leakage.

In [1]:
import os, sys, subprocess, json
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

os.makedirs("work/outputs", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Capstone Dataset Loaded. Shape: {df.shape}")

Capstone Dataset Loaded. Shape: (30000, 45)


In [2]:
feature_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
X = df[feature_cols].fillna(0)
y = df["is_declining_label"]

split_idx = int(len(df) * 0.8)
X_train, X_eval = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_eval = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train size: {len(X_train)} rows | Eval size: {len(X_eval)} rows")

Train size: 24000 rows | Eval size: 6000 rows


In [3]:
# 1. Heuristic Baseline Score
max_stale = X_eval["days_since_last_update"].max()
max_imp = X_eval["impressions_90d"].max()
baseline_score = 0.6 * (X_eval["days_since_last_update"] / max_stale) + 0.4 * (X_eval["impressions_90d"] / max_imp)
baseline_p20 = y_eval.loc[baseline_score.nlargest(20).index].mean()

# 2. Random Forest Model
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
rf_probs = pd.Series(rf.predict_proba(X_eval)[:, 1], index=X_eval.index)
ml_p20 = y_eval.loc[rf_probs.nlargest(20).index].mean()

print(f"Heuristic Baseline Precision@20: {baseline_p20:.3f}")
print(f"Random Forest ML Precision@20:   {ml_p20:.3f}")
print(f"Observed Precision Lift:         +{(ml_p20 - baseline_p20):.3f}")

Heuristic Baseline Precision@20: 0.550
Random Forest ML Precision@20:   0.950
Observed Precision Lift:         +0.400


In [4]:
metrics = {
    "baseline_precision_at_20": float(baseline_p20),
    "ml_precision_at_20": float(ml_p20),
    "precision_lift": float(ml_p20 - baseline_p20),
    "features_used": feature_cols
}

with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Receipts exported to work/outputs/capstone_metrics.json")

Receipts exported to work/outputs/capstone_metrics.json
